In [1]:
!nvidia-smi

Sun Oct 11 05:45:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile kolamnet_optuna.py
"""
Kolam motif classification: pretrained backbone + CBAM, with SMOTE class
balancing, Optuna hyperparameter search, early stopping, and metric plots.

Supported backbones (chosen with --model_name):
  efficientnet_b4       CNN, 380 x 380 input (default; KolamNet)
  vgg16                 CNN, 224 x 224 input (use --img_size 224)
  vit_base_patch16_224  Vision Transformer, 224 x 224 input (use --img_size 224)

Reference: KolamNet: An Attention-Based Model for Kolam Classification (2022).
CBAM is inserted on the final feature map of the backbone, before global
pooling and the classifier ("pre-final layers" in the paper). For ViT, the
patch tokens are rearranged into their 14 x 14 grid before CBAM.

Expected data layout (one sub-folder per class, any image format PIL can read):
    data/
      animal/     *.jpg
      footprint/  *.jpg
      geometric/  *.jpg
      ...

Usage (terminal):
    pip install torch torchvision timm optuna scikit-learn matplotlib pandas
    python kolamnet_optuna.py --data_dir data --out_dir runs/kolamnet

    # ablation: same pipeline without CBAM (plain backbone)
    python kolamnet_optuna.py --data_dir data --out_dir runs/b4_only --use_cbam false

    # Vision Transformer backbone instead of EfficientNet (224 x 224 input)
    python kolamnet_optuna.py --data_dir data --out_dir runs/vit --model_name vit_base_patch16_224 --img_size 224

    # VGG-16 backbone (224 x 224 input)
    python kolamnet_optuna.py --data_dir data --out_dir runs/vgg16 --model_name vgg16 --img_size 224

    # comparison: same pipeline without SMOTE (imbalanced training set)
    python kolamnet_optuna.py --data_dir data --out_dir runs/no_smote --use_smote false

    # skip tuning, train once with DEFAULT_HP
    python kolamnet_optuna.py --data_dir data --n_trials 0

Cells are marked with "# %%" so the file also runs cell by cell in VS Code or
Jupyter / Colab. Change the Config defaults below when running as a notebook.

Pipeline
  1. Stratified split: train / val / test. The test split is used once, at the end.
  2. SMOTE balances the TRAINING split only: for every class smaller than the
     largest one, synthetic images are created by interpolating between an image
     and one of its k nearest neighbours of the same class (in pixel space), until
     all classes have as many training images as the largest class. Validation and
     test images are never touched. The synthetic images are cached on disk and
     reused by every Optuna trial and the final training.
  3. Optuna tunes the hyperparameters on the validation split (objective: macro-F1).
     Unpromising trials are pruned early (MedianPruner).
  4. The best hyperparameters are used to train the final model, with early
     stopping on validation loss and the best weights restored.
  5. The final model is evaluated on the test split, and all plots are saved.
"""

# %% Imports
import argparse
import copy
import json
import math
import random
import shutil
import tempfile
import time
from functools import lru_cache
from dataclasses import dataclass, fields, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from torchvision.datasets.folder import default_loader
import timm
import optuna
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_recall_fscore_support)
import matplotlib.pyplot as plt


# %% Configuration
@dataclass
class Config:
    data_dir: str = "data"
    out_dir: str = "runs/kolamnet"
    model_name: str = "efficientnet_b4"  # or vgg16 / vit_base_patch16_224 (use --img_size 224)
    pretrained: bool = True          # ImageNet weights (downloaded by timm on first run)
    use_cbam: bool = True            # False = plain backbone (no CBAM) for the ablation
    img_size: int = 380              # 380 for EfficientNet-B4; use 224 for vgg16 and ViT
    test_size: float = 0.2           # held-out test fraction (paper: 80/20)
    val_size: float = 0.2            # fraction of the training split used for validation
    use_smote: bool = True           # balance the training split with SMOTE
    smote_k: int = 5                 # SMOTE nearest neighbours
    smote_cache: str = ""            # where synthetic images are stored ("" = system temp folder)
    n_trials: int = 20               # Optuna trials (0 = skip tuning, use DEFAULT_HP)
    tune_epochs: int = 15            # max epochs per trial
    tune_patience: int = 4           # early-stopping patience during tuning
    final_epochs: int = 100          # max epochs for the final model
    patience: int = 10               # early-stopping patience for the final model
    batch_sizes: tuple = (8, 16)     # batch sizes Optuna may choose from (lower if you hit OOM)
    lr_max: float = 1e-3             # upper end of Optuna's learning-rate range (try 3e-4 if trials diverge)
    max_grad_norm: float = 1.0       # gradient clipping threshold (0 = no clipping)
    num_workers: int = 2
    seed: int = 42


# Used when n_trials = 0
DEFAULT_HP = {
    "lr": 1e-4, "weight_decay": 1e-5, "batch_size": 8, "dropout": 0.3,
    "cbam_reduction": 16, "optimizer": "adam", "label_smoothing": 0.0,
}


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# %% Data
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


class RandomRot90:
    """Rotate by 0/90/180/270 degrees. Kolams keep their class under these rotations."""
    def __call__(self, img):
        k = random.randint(0, 3)
        return img.rotate(90 * k, expand=True) if k else img


@lru_cache(maxsize=None)
def get_norm(model_name: str):
    """Mean/std the pretrained backbone expects (ImageNet values for EfficientNet, 0.5 for ViT)."""
    try:
        m = timm.create_model(model_name, pretrained=False, num_classes=0)
        c = timm.data.resolve_model_data_config(m)
        return tuple(c["mean"]), tuple(c["std"])
    except Exception:
        return IMAGENET_MEAN, IMAGENET_STD


def build_transforms(img_size: int, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    train_tf = transforms.Compose([
        RandomRot90(),
        transforms.RandomResizedCrop(img_size, scale=(0.75, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    # Resize the whole image (no centre crop) so the edges of the kolam are kept
    eval_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    return train_tf, eval_tf


def make_splits(cfg: Config):
    base = datasets.ImageFolder(cfg.data_dir)
    targets = np.array(base.targets)
    idx = np.arange(len(targets))
    trainval_idx, test_idx = train_test_split(
        idx, test_size=cfg.test_size, stratify=targets, random_state=cfg.seed)
    train_idx, val_idx = train_test_split(
        trainval_idx, test_size=cfg.val_size, stratify=targets[trainval_idx],
        random_state=cfg.seed)
    return base.classes, base.samples, targets, train_idx, val_idx, test_idx


class RecordsDataset(Dataset):
    """A list of (image_path, label) pairs; used for real and SMOTE images alike."""
    def __init__(self, records, transform):
        self.records, self.transform = records, transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        path, label = self.records[i]
        return self.transform(default_loader(path)), label


def make_loaders(cfg, train_records, val_records, test_records, batch_size):
    train_tf, eval_tf = build_transforms(cfg.img_size, *get_norm(cfg.model_name))
    pin = torch.cuda.is_available()
    common = dict(num_workers=cfg.num_workers, pin_memory=pin)
    train_loader = DataLoader(RecordsDataset(train_records, train_tf), batch_size=batch_size,
                              shuffle=True, drop_last=len(train_records) > batch_size, **common)
    val_loader = DataLoader(RecordsDataset(val_records, eval_tf), batch_size=batch_size * 2,
                            shuffle=False, **common)
    test_loader = DataLoader(RecordsDataset(test_records, eval_tf), batch_size=batch_size * 2,
                             shuffle=False, **common)
    return train_loader, val_loader, test_loader


# %% SMOTE (training split only)
def load_resized(path, size):
    with Image.open(path) as im:
        return np.asarray(im.convert("RGB").resize((size, size), Image.BILINEAR), dtype=np.uint8)


def build_smote_images(cfg, train_records, n_classes, cache_dir):
    """
    Pixel-level SMOTE on the training images.

    For each class smaller than the largest class, repeat until it reaches the
    largest class's count:
      - pick a random training image x_i of that class,
      - pick one of its k nearest neighbours x_j (same class, Euclidean pixel distance),
      - create x_new = x_i + lambda * (x_j - x_i), with lambda ~ Uniform(0, 1).
    Returns a list of (path, label) for the synthetic images.
    """
    labels = np.array([y for _, y in train_records])
    counts = np.bincount(labels, minlength=n_classes)
    target = int(counts.max())
    meta = {"img_size": cfg.img_size, "seed": cfg.seed, "k": cfg.smote_k,
            "train_files": sorted(p for p, _ in train_records)}
    marker = cache_dir / "smote_meta.json"
    if marker.exists():
        saved = json.load(open(marker))
        if all(saved.get(k) == v for k, v in meta.items()) and \
                all(Path(p).exists() for p, _ in saved["synthetic"]):
            print(f"  reusing {len(saved['synthetic'])} cached SMOTE images from {cache_dir}")
            return [tuple(r) for r in saved["synthetic"]]
    shutil.rmtree(cache_dir, ignore_errors=True)
    cache_dir.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(cfg.seed)
    synthetic = []
    for c in range(n_classes):
        need = target - int(counts[c])
        if need <= 0:
            continue
        paths_c = [p for p, y in train_records if y == c]
        if len(paths_c) < 2:
            raise ValueError(f"Class {c} has fewer than 2 training images; SMOTE needs at least 2.")
        X = np.stack([load_resized(p, cfg.img_size) for p in paths_c]).astype(np.float32)
        flat = X.reshape(len(X), -1)
        k = min(cfg.smote_k, len(X) - 1)
        neighbours = NearestNeighbors(n_neighbors=k + 1).fit(flat).kneighbors(
            flat, return_distance=False)[:, 1:]        # drop column 0 (the image itself)
        out_c = cache_dir / f"class_{c:02d}"
        out_c.mkdir(parents=True, exist_ok=True)
        for n in range(need):
            i = rng.integers(len(X))
            j = neighbours[i, rng.integers(k)]
            lam = rng.random()
            new = X[i] + lam * (X[j] - X[i])
            path = out_c / f"smote_{n:05d}.jpg"
            Image.fromarray(np.clip(np.rint(new), 0, 255).astype(np.uint8)).save(path, quality=95)
            synthetic.append((str(path), c))
        print(f"  class {c}: {counts[c]} real + {need} synthetic = {target}")
        del X, flat
    json.dump({**meta, "synthetic": synthetic}, open(marker, "w"))
    return synthetic


def plot_smote_distribution(classes, before, after, path):
    x = np.arange(len(classes))
    w = 0.4
    fig, ax = plt.subplots(figsize=(max(8, len(classes) * 1.2), 4.5))
    ax.bar(x - w / 2, before, w, label="Original training set")
    ax.bar(x + w / 2, after, w, label="After SMOTE")
    ax.set_xticks(x, classes, rotation=35, ha="right")
    ax.set_ylabel("Training images")
    ax.set_title("Training class distribution before and after SMOTE")
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def plot_smote_examples(classes, synthetic, path, per_class=4, seed=0):
    rng = random.Random(seed)
    by_class = {}
    for p, y in synthetic:
        by_class.setdefault(y, []).append(p)
    rows = sorted(by_class)
    if not rows:
        return
    fig, axes = plt.subplots(len(rows), per_class, figsize=(per_class * 2.2, len(rows) * 2.3),
                             squeeze=False)
    for r, c in enumerate(rows):
        picks = rng.sample(by_class[c], min(per_class, len(by_class[c])))
        for k in range(per_class):
            ax = axes[r][k]
            ax.axis("off")
            if k < len(picks):
                ax.imshow(Image.open(picks[k]))
        axes[r][0].set_title(classes[c], fontsize=9, loc="left")
    fig.suptitle("Examples of SMOTE-generated training images")
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)


# %% Model: backbone (EfficientNet / VGG / ViT) + CBAM
class ChannelAttention(nn.Module):
    """'What' to attend to: re-weights feature channels."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, hidden, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1, bias=False),
        )

    def forward(self, x):
        avg = self.mlp(F.adaptive_avg_pool2d(x, 1))
        mx = self.mlp(F.adaptive_max_pool2d(x, 1))
        return x * torch.sigmoid(avg + mx)


class SpatialAttention(nn.Module):
    """'Where' to attend to: re-weights spatial positions."""
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)

    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx, _ = x.max(dim=1, keepdim=True)
        return x * torch.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))


class CBAM(nn.Module):
    """Channel attention followed by spatial attention (Woo et al., 2018)."""
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        # Run attention in float32: under fp16 autocast the channel-attention MLP can
        # overflow (fp16 max is about 65,500) and turn the whole forward pass into NaN
        with torch.autocast(device_type=x.device.type, enabled=False):
            return self.sa(self.ca(x.float()))


class KolamNet(nn.Module):
    """
    Backbone + (optional) CBAM + classifier.

    CNN backbones (e.g. efficientnet_b4, vgg16) output their last convolutional feature
    map (B, C, H, W); CBAM is applied to it, followed by global average pooling. For VGG,
    the original fully connected layers (fc6/fc7) are dropped and replaced by this head.

    ViT backbones (e.g. vit_base_patch16_224) output tokens (B, 1 + H*W, C): one CLS
    token plus one token per image patch. With CBAM, the patch tokens are rearranged
    into their H x W grid (14 x 14 for 224-pixel images with 16-pixel patches), CBAM is
    applied to that grid, and the grid is average-pooled. Without CBAM, the standard
    ViT classifier input (the CLS token) is used.
    """
    def __init__(self, n_classes, model_name="efficientnet_b4", pretrained=True,
                 use_cbam=True, cbam_reduction=16, dropout=0.3, img_size=None):
        super().__init__()
        self.is_vit = model_name.startswith(("vit_", "deit", "eva", "beit"))
        if self.is_vit:
            kwargs = {"img_size": img_size} if img_size else {}
            self.backbone = timm.create_model(model_name, pretrained=pretrained,
                                              num_classes=0, **kwargs)
            self.prefix = self.backbone.num_prefix_tokens          # 1 CLS token
            self.grid = self.backbone.patch_embed.grid_size         # (14, 14)
        else:
            # CNN: use the last convolutional feature map (B, C, H, W) from forward_features.
            self.backbone = timm.create_model(model_name, pretrained=pretrained,
                                              num_classes=0, global_pool="")
            # VGG's fully connected fc6/fc7 layers ("pre_logits") are not used: CBAM needs the
            # convolutional feature map, and the new classifier head replaces them.
            if hasattr(self.backbone, "pre_logits"):
                self.backbone.pre_logits = nn.Identity()
        c = self.backbone.num_features
        self.use_cbam = use_cbam
        self.cbam = CBAM(c, cbam_reduction) if use_cbam else nn.Identity()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(c, n_classes)

    def forward(self, x):
        if self.is_vit:
            tokens = self.backbone.forward_features(x)              # (B, 1 + H*W, C)
            if not self.use_cbam:
                return self.fc(self.dropout(tokens[:, 0]))          # CLS token
            patches = tokens[:, self.prefix:]                       # (B, H*W, C)
            b, n, c = patches.shape
            feats = patches.transpose(1, 2).reshape(b, c, *self.grid)   # (B, C, H, W)
        else:
            feats = self.backbone.forward_features(x)               # (B, C, H, W)
        feats = self.cbam(feats)
        return self.fc(self.dropout(self.pool(feats).flatten(1)))


# %% Training utilities
def make_scaler(device):
    enabled = device.type == "cuda"
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):   # older PyTorch
        return torch.cuda.amp.GradScaler(enabled=enabled)


def run_epoch(model, loader, criterion, device, optimizer=None, scaler=None, max_grad_norm=1.0):
    training = optimizer is not None
    model.train(training)
    use_amp = device.type == "cuda"
    total_loss, n_seen, bad_batches = 0.0, 0, 0
    y_true, y_pred = [], []
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(x)
            loss = criterion(logits.float(), y)          # loss in float32, not fp16

            if training:
                optimizer.zero_grad(set_to_none=True)
                if not torch.isfinite(loss):              # skip a NaN/inf batch instead of
                    bad_batches += 1                      # letting it corrupt the weights
                    continue
                scaler.scale(loss).backward()
                if max_grad_norm and max_grad_norm > 0:
                    scaler.unscale_(optimizer)            # unscale so clipping sees true gradient sizes
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                scaler.step(optimizer)
                scaler.update()

            total_loss += loss.item() * x.size(0)
            n_seen += x.size(0)
            y_true.append(y.cpu())
            y_pred.append(logits.argmax(1).cpu())

    if n_seen == 0:                                       # every batch failed
        return {"loss": float("nan"), "acc": 0.0, "f1": 0.0, "bad_batches": bad_batches}
    y_true, y_pred = torch.cat(y_true).numpy(), torch.cat(y_pred).numpy()
    return {
        "loss": total_loss / n_seen,
        "acc": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "bad_batches": bad_batches,
    }


def fit(model, train_loader, val_loader, hp, device, max_epochs, patience, trial=None,
        max_grad_norm=1.0):
    """Train with early stopping on validation loss; restores the best weights."""
    criterion = nn.CrossEntropyLoss(label_smoothing=hp["label_smoothing"])
    opt_cls = torch.optim.AdamW if hp["optimizer"] == "adamw" else torch.optim.Adam
    optimizer = opt_cls(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min",
                                                           factor=0.5, patience=3)
    scaler = make_scaler(device)

    history = {k: [] for k in ["train_loss", "val_loss", "train_acc", "val_acc",
                               "train_f1", "val_f1", "lr"]}
    best_loss, best_state, best_epoch, wait = float("inf"), None, 0, 0

    for epoch in range(max_epochs):
        tr = run_epoch(model, train_loader, criterion, device, optimizer, scaler, max_grad_norm)
        va = run_epoch(model, val_loader, criterion, device)

        # Divergence check: NaN validation loss, or more than 10% of training batches were NaN
        diverged = (not math.isfinite(va["loss"])) or tr["bad_batches"] > 0.1 * len(train_loader)
        if diverged:
            msg = (f"training diverged at epoch {epoch + 1} "
                   f"({tr['bad_batches']} NaN training batches, val loss {va['loss']})")
            print("  " + msg)
            if trial is not None:
                raise optuna.TrialPruned(msg)         # Optuna skips this trial and moves on
            if best_state is None:
                raise RuntimeError(msg + " before any valid epoch - lower the learning rate")
            break                                     # final model: keep the last good weights

        scheduler.step(va["loss"])

        for split, m in (("train", tr), ("val", va)):
            for k in ("loss", "acc", "f1"):
                history[f"{split}_{k}"].append(m[k])
        history["lr"].append(optimizer.param_groups[0]["lr"])
        skipped = f" | skipped {tr['bad_batches']} NaN batches" if tr["bad_batches"] else ""
        print(f"  epoch {epoch + 1:3d} | train loss {tr['loss']:.4f} acc {tr['acc']:.3f} "
              f"F1 {tr['f1']:.3f} | val loss {va['loss']:.4f} acc {va['acc']:.3f} "
              f"F1 {va['f1']:.3f}{skipped}")

        if va["loss"] < best_loss - 1e-4:
            best_loss, best_epoch, wait = va["loss"], epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1

        if trial is not None:
            trial.report(va["f1"], epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        if wait >= patience:
            print(f"  early stopping at epoch {epoch + 1} (best epoch {best_epoch + 1})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history, best_epoch


@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    y_true, y_pred, probs = [], [], []
    for x, y in loader:
        logits = model(x.to(device))
        p = logits.float().softmax(1).cpu()
        probs.append(p)
        y_pred.append(p.argmax(1))
        y_true.append(y)
    return torch.cat(y_true).numpy(), torch.cat(y_pred).numpy(), torch.cat(probs).numpy()


# %% Optuna
def suggest_hp(trial, cfg):
    return {
        "lr": trial.suggest_float("lr", 1e-5, cfg.lr_max, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "batch_size": trial.suggest_categorical("batch_size", list(cfg.batch_sizes)),
        "dropout": trial.suggest_float("dropout", 0.1, 0.5),
        "cbam_reduction": trial.suggest_categorical("cbam_reduction", [8, 16, 32]),
        "optimizer": trial.suggest_categorical("optimizer", ["adam", "adamw"]),
        "label_smoothing": trial.suggest_float("label_smoothing", 0.0, 0.1),
    }


def build_model(cfg, hp, n_classes, device):
    return KolamNet(n_classes, cfg.model_name, cfg.pretrained, cfg.use_cbam,
                    hp["cbam_reduction"], hp["dropout"], img_size=cfg.img_size).to(device)


def run_study(cfg, n_classes, train_records, val_records, test_records, device, out):
    study = optuna.create_study(
        study_name="kolamnet",
        direction="maximize",
        storage=f"sqlite:///{out / 'optuna.db'}",   # resumable if the session disconnects
        load_if_exists=True,
        # TPE starts guided search after a few random trials (its default of 10 would make
        # short studies pure random search)
        sampler=optuna.samplers.TPESampler(seed=cfg.seed, n_startup_trials=max(3, cfg.n_trials // 3)),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3),
    )

    def objective(trial):
        set_seed(cfg.seed + trial.number)
        hp = suggest_hp(trial, cfg)
        print(f"\nTrial {trial.number}: {hp}")
        train_loader, val_loader, _ = make_loaders(cfg, train_records, val_records, test_records,
                                                   hp["batch_size"])
        model = build_model(cfg, hp, n_classes, device)
        try:
            model, _, _ = fit(model, train_loader, val_loader, hp, device,
                              cfg.tune_epochs, cfg.tune_patience, trial, cfg.max_grad_norm)
        except torch.cuda.OutOfMemoryError:
            del model
            torch.cuda.empty_cache()
            print("  out of GPU memory - trial pruned (reduce --batch_sizes or --img_size)")
            raise optuna.TrialPruned()
        y_true, y_pred, _ = predict(model, val_loader, device)
        return f1_score(y_true, y_pred, average="macro", zero_division=0)

    done = len([t for t in study.trials if t.state.is_finished()])
    remaining = max(cfg.n_trials - done, 0)
    if remaining:
        # catch: one crashed trial is recorded as FAIL and the search continues
        study.optimize(objective, n_trials=remaining, catch=(RuntimeError, ValueError, TypeError))
    return study


# %% Plots
def plot_history(history, best_epoch, path):
    epochs = np.arange(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, key, title in zip(axes, ["loss", "acc", "f1"], ["Loss", "Accuracy", "Macro-F1"]):
        ax.plot(epochs, history[f"train_{key}"], label="train")
        ax.plot(epochs, history[f"val_{key}"], label="validation")
        ax.axvline(best_epoch + 1, color="grey", ls="--", lw=1, label="restored epoch")
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax.grid(alpha=0.3)
    axes[0].legend()
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def plot_confusion(y_true, y_pred, classes, path):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(classes)))
    cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    for ax, mat, title, fmt in ((axes[0], cm, "Counts", "d"),
                                (axes[1], cm_norm, "Row-normalised (recall)", ".2f")):
        im = ax.imshow(mat, cmap="Blues")
        ax.grid(False)
        ax.set_xticks(range(len(classes)), classes, rotation=45, ha="right")
        ax.set_yticks(range(len(classes)), classes)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.set_title(title)
        thresh = mat.max() / 2
        for i in range(len(classes)):
            for j in range(len(classes)):
                ax.text(j, i, format(mat[i, j], fmt), ha="center", va="center",
                        color="white" if mat[i, j] > thresh else "black", fontsize=9)
        fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def plot_per_class(y_true, y_pred, classes, path):
    p, r, f, s = precision_recall_fscore_support(
        y_true, y_pred, labels=range(len(classes)), zero_division=0)
    x = np.arange(len(classes))
    w = 0.27
    fig, ax = plt.subplots(figsize=(max(7, len(classes) * 1.3), 4.5))
    ax.bar(x - w, p, w, label="Precision")
    ax.bar(x, r, w, label="Recall")
    ax.bar(x + w, f, w, label="F1")
    ax.set_xticks(x, [f"{c}\n(n={n})" for c, n in zip(classes, s)], fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.set_title("Per-class test metrics")
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def plot_optuna(study, out):
    from optuna.visualization import matplotlib as ovm
    complete = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    plots = [("optuna_history.png", ovm.plot_optimization_history)]
    if len(complete) >= 2:
        plots.append(("optuna_param_importance.png", ovm.plot_param_importances))
    for name, fn in plots:
        try:
            # Optuna's plots switch matplotlib to the ggplot style; keep that change local
            with plt.style.context("default"):
                ax = fn(study)
            fig = ax.figure if hasattr(ax, "figure") else ax[0].figure
            fig.set_size_inches(8, 5)
            fig.tight_layout()
            fig.savefig(out / name, dpi=150)
            plt.close(fig)
        except Exception as e:   # plotting should never kill a finished run
            print(f"Could not create {name}: {e}")


# %% Main
def main(cfg: Config):
    set_seed(cfg.seed)
    out = Path(cfg.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    mean, std = get_norm(cfg.model_name)
    print(f"Backbone: {cfg.model_name} | image size {cfg.img_size} | CBAM: {cfg.use_cbam} | "
          f"normalisation mean {mean} std {std}")

    classes, samples, targets, train_idx, val_idx, test_idx = make_splits(cfg)
    n_classes = len(classes)
    print("Classes:", classes)
    for name, idx in (("train", train_idx), ("val", val_idx), ("test", test_idx)):
        print(f"  {name:5s} {len(idx):5d} images | per class: "
              f"{np.bincount(targets[idx], minlength=n_classes).tolist()}")
    json.dump({"classes": classes, "train": train_idx.tolist(), "val": val_idx.tolist(),
               "test": test_idx.tolist()}, open(out / "splits.json", "w"))

    train_records = [samples[i] for i in train_idx]
    val_records = [samples[i] for i in val_idx]
    test_records = [samples[i] for i in test_idx]

    # 1. SMOTE on the training split only
    if cfg.use_smote:
        print("\nBalancing the training split with SMOTE")
        cache_dir = Path(cfg.smote_cache) if cfg.smote_cache else \
            Path(tempfile.gettempdir()) / f"kolam_smote_{cfg.img_size}_{cfg.seed}"
        synthetic = build_smote_images(cfg, train_records, n_classes, cache_dir)
        before = np.bincount(targets[train_idx], minlength=n_classes)
        train_records = train_records + synthetic
        after = np.bincount([y for _, y in train_records], minlength=n_classes)
        print(f"  training set: {int(before.sum())} real + {len(synthetic)} synthetic = "
              f"{int(after.sum())} | per class: {after.tolist()}")
        plot_smote_distribution(classes, before, after, out / "smote_class_distribution.png")
        plot_smote_examples(classes, synthetic, out / "smote_examples.png", seed=cfg.seed)
    else:
        print("\nSMOTE disabled: training on the original (imbalanced) split")

    # 2. Hyperparameter search
    if cfg.n_trials > 0:
        study = run_study(cfg, n_classes, train_records, val_records, test_records, device, out)
        study.trials_dataframe().to_csv(out / "optuna_trials.csv", index=False)
        states = [t.state.name for t in study.trials]
        print(f"\nTrials: {states.count('COMPLETE')} complete, {states.count('PRUNED')} pruned, "
              f"{states.count('FAIL')} failed")
        if states.count("COMPLETE") == 0:
            best_hp = dict(DEFAULT_HP)
            print("No trial completed (all pruned or diverged); using DEFAULT_HP:", best_hp)
        else:
            best_hp = dict(study.best_params)
            print(f"Best trial {study.best_trial.number}: val macro-F1 = {study.best_value:.4f}")
            print("Best hyperparameters:", best_hp)
            plot_optuna(study, out)
    else:
        best_hp = dict(DEFAULT_HP)
        print("Skipping tuning; using DEFAULT_HP:", best_hp)
    json.dump(best_hp, open(out / "best_hyperparameters.json", "w"), indent=2)

    # 3. Final training with the best hyperparameters
    print("\nTraining final model")
    set_seed(cfg.seed)
    train_loader, val_loader, test_loader = make_loaders(
        cfg, train_records, val_records, test_records, best_hp["batch_size"])
    model = build_model(cfg, best_hp, n_classes, device)
    t0 = time.time()
    model, history, best_epoch = fit(model, train_loader, val_loader, best_hp, device,
                                     cfg.final_epochs, cfg.patience,
                                     max_grad_norm=cfg.max_grad_norm)
    train_minutes = (time.time() - t0) / 60
    torch.save({"state_dict": model.state_dict(), "classes": classes,
                "hyperparameters": best_hp, "config": asdict(cfg)}, out / "kolamnet_best.pt")
    pd.DataFrame(history).rename_axis("epoch").to_csv(out / "history.csv")

    # 4. Test evaluation (the only time the test split is used; real images only)
    y_true, y_pred, _ = predict(model, test_loader, device)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    pw, rw, fw, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted",
                                                    zero_division=0)
    metrics = {
        "test_accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": p, "macro_recall": r, "macro_f1": f,
        "weighted_precision": pw, "weighted_recall": rw, "weighted_f1": fw,
        "n_test_images": int(len(y_true)),
        "n_train_images": len(train_records),
        "restored_epoch": best_epoch + 1,
        "epochs_run": len(history["train_loss"]),
        "train_minutes": round(train_minutes, 1),
        "use_cbam": cfg.use_cbam,
        "use_smote": cfg.use_smote,
    }
    report = classification_report(y_true, y_pred, labels=range(n_classes),
                                   target_names=classes, digits=3, zero_division=0)
    json.dump(metrics, open(out / "test_metrics.json", "w"), indent=2)
    (out / "classification_report.txt").write_text(report)
    print("\nTest metrics:", json.dumps(metrics, indent=2))
    print(report)

    # 5. Plots
    plot_history(history, best_epoch, out / "training_curves.png")
    plot_confusion(y_true, y_pred, classes, out / "confusion_matrix.png")
    plot_per_class(y_true, y_pred, classes, out / "per_class_metrics.png")
    print(f"All outputs saved to {out.resolve()}")
    return model, metrics


def parse_args() -> Config:
    parser = argparse.ArgumentParser(description="Kolam classifier (EfficientNet / VGG / ViT + CBAM) with SMOTE and Optuna")
    for f in fields(Config):
        default = f.default
        if isinstance(default, bool):
            parser.add_argument(f"--{f.name}", default=default,
                                type=lambda s: s.lower() in ("1", "true", "yes", "y"))
        elif isinstance(default, tuple):
            parser.add_argument(f"--{f.name}", default=default, type=int, nargs="+")
        else:
            parser.add_argument(f"--{f.name}", default=default, type=type(default))
    args, _ = parser.parse_known_args()   # parse_known_args also works inside Jupyter
    values = vars(args)
    values["batch_sizes"] = tuple(values["batch_sizes"])
    return Config(**values)


if __name__ == "__main__":
    main(parse_args())

Writing kolamnet_optuna.py


In [3]:
!python -u kolamnet_optuna.py --data_dir "/kaggle/input/datasets/skuvilan/kolam-dataset/Kolam_Project" --out_dir run_vgg_v2 --model_name vgg16 --img_size 224 --n_trials 8 --tune_epochs 5 --num_workers 4

Device: cuda
Backbone: vgg16 | image size 224 | CBAM: True | normalisation mean (0.485, 0.456, 0.406) std (0.229, 0.224, 0.225)
Classes: ['Artifical Kolam', 'Digital Art Kolam', 'Mandala Kolam', 'Peakcock Kolam', 'Pushpa Kolam_Pookalam', 'Rangoli Kolam', 'Sikku_Pulli Kolam_Geometric Patterns', 'Traditional Kolam_Colourful', 'Traditional Kolam_Dominant White Colour']
  train  4040 images | per class: [136, 373, 1364, 328, 230, 286, 395, 398, 530]
  val    1011 images | per class: [34, 93, 341, 82, 58, 72, 99, 100, 132]
  test   1263 images | per class: [42, 117, 427, 103, 72, 89, 123, 124, 166]

Balancing the training split with SMOTE
  class 0: 136 real + 1228 synthetic = 1364
  class 1: 373 real + 991 synthetic = 1364
  class 3: 328 real + 1036 synthetic = 1364
  class 4: 230 real + 1134 synthetic = 1364
  class 5: 286 real + 1078 synthetic = 1364
  class 6: 395 real + 969 synthetic = 1364
  class 7: 398 real + 966 synthetic = 1364
  class 8: 530 real + 834 synthetic = 1364
  training

In [4]:
!zip -r run_vgg_v2.zip run_vgg_v2

  adding: run_vgg_v2/ (stored 0%)
  adding: run_vgg_v2/optuna.db (deflated 96%)
  adding: run_vgg_v2/optuna_trials.csv (deflated 54%)
  adding: run_vgg_v2/test_metrics.json (deflated 48%)
  adding: run_vgg_v2/per_class_metrics.png (deflated 20%)
  adding: run_vgg_v2/classification_report.txt (deflated 67%)
  adding: run_vgg_v2/smote_class_distribution.png (deflated 14%)
  adding: run_vgg_v2/history.csv (deflated 56%)
  adding: run_vgg_v2/optuna_history.png (deflated 18%)
  adding: run_vgg_v2/confusion_matrix.png (deflated 10%)
  adding: run_vgg_v2/optuna_param_importance.png (deflated 21%)
  adding: run_vgg_v2/smote_examples.png (deflated 0%)
  adding: run_vgg_v2/training_curves.png (deflated 7%)
  adding: run_vgg_v2/best_hyperparameters.json (deflated 27%)
  adding: run_vgg_v2/splits.json (deflated 60%)
  adding: run_vgg_v2/kolamnet_best.pt (deflated 7%)


# Without Smote 

In [2]:
%%writefile kolamnet_optuna.py
"""
Kolam motif classification: pretrained backbone + CBAM, with SMOTE class
balancing, Optuna hyperparameter search, early stopping, and metric plots.

Supported backbones (chosen with --model_name):
  efficientnet_b4       CNN, 380 x 380 input (default; KolamNet)
  vgg16                 CNN, 224 x 224 input (use --img_size 224)
  vit_base_patch16_224  Vision Transformer, 224 x 224 input (use --img_size 224)

Reference: KolamNet: An Attention-Based Model for Kolam Classification (2022).
CBAM is inserted on the final feature map of the backbone, before global
pooling and the classifier ("pre-final layers" in the paper). For ViT, the
patch tokens are rearranged into their 14 x 14 grid before CBAM.

Expected data layout (one sub-folder per class, any image format PIL can read):
    data/
      animal/     *.jpg
      footprint/  *.jpg
      geometric/  *.jpg
      ...

Usage (terminal):
    pip install torch torchvision timm optuna scikit-learn matplotlib pandas
    python kolamnet_optuna.py --data_dir data --out_dir runs/kolamnet

    # ablation: same pipeline without CBAM (plain backbone)
    python kolamnet_optuna.py --data_dir data --out_dir runs/b4_only --use_cbam false

    # Vision Transformer backbone instead of EfficientNet (224 x 224 input)
    python kolamnet_optuna.py --data_dir data --out_dir runs/vit --model_name vit_base_patch16_224 --img_size 224

    # VGG-16 backbone (224 x 224 input)
    python kolamnet_optuna.py --data_dir data --out_dir runs/vgg16 --model_name vgg16 --img_size 224

    # comparison: same pipeline without SMOTE (imbalanced training set)
    python kolamnet_optuna.py --data_dir data --out_dir runs/no_smote --use_smote false

    # skip tuning, train once with DEFAULT_HP
    python kolamnet_optuna.py --data_dir data --n_trials 0

Cells are marked with "# %%" so the file also runs cell by cell in VS Code or
Jupyter / Colab. Change the Config defaults below when running as a notebook.

Pipeline
  1. Stratified split: train / val / test. The test split is used once, at the end.
  2. SMOTE balances the TRAINING split only: for every class smaller than the
     largest one, synthetic images are created by interpolating between an image
     and one of its k nearest neighbours of the same class (in pixel space), until
     all classes have as many training images as the largest class. Validation and
     test images are never touched. The synthetic images are cached on disk and
     reused by every Optuna trial and the final training.
  3. Optuna tunes the hyperparameters on the validation split (objective: macro-F1).
     Unpromising trials are pruned early (MedianPruner).
  4. The best hyperparameters are used to train the final model, with early
     stopping on validation loss and the best weights restored.
  5. The final model is evaluated on the test split, and all plots are saved.
"""

# %% Imports
import argparse
import copy
import json
import math
import random
import shutil
import tempfile
import time
from functools import lru_cache
from dataclasses import dataclass, fields, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from torchvision.datasets.folder import default_loader
import timm
import optuna
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_recall_fscore_support)
import matplotlib.pyplot as plt


# %% Configuration
@dataclass
class Config:
    data_dir: str = "data"
    out_dir: str = "runs/kolamnet"
    model_name: str = "efficientnet_b4"  # or vgg16 / vit_base_patch16_224 (use --img_size 224)
    pretrained: bool = True          # ImageNet weights (downloaded by timm on first run)
    use_cbam: bool = True            # False = plain backbone (no CBAM) for the ablation
    img_size: int = 380              # 380 for EfficientNet-B4; use 224 for vgg16 and ViT
    test_size: float = 0.2           # held-out test fraction (paper: 80/20)
    val_size: float = 0.2            # fraction of the training split used for validation
    use_smote: bool = False          # balance the training split with SMOTE
    smote_k: int = 5                 # SMOTE nearest neighbours
    smote_cache: str = ""            # where synthetic images are stored ("" = system temp folder)
    n_trials: int = 0                # Optuna trials (0 = skip tuning, use DEFAULT_HP)
    tune_epochs: int = 15            # max epochs per trial
    tune_patience: int = 4           # early-stopping patience during tuning
    final_epochs: int = 100          # max epochs for the final model
    patience: int = 10               # early-stopping patience for the final model
    batch_sizes: tuple = (8, 16)     # batch sizes Optuna may choose from (lower if you hit OOM)
    lr_max: float = 1e-3             # upper end of Optuna's learning-rate range (try 3e-4 if trials diverge)
    max_grad_norm: float = 1.0       # gradient clipping threshold (0 = no clipping)
    num_workers: int = 2
    seed: int = 42


# Used when n_trials = 0
# Fixed to the hyperparameters Optuna selected for VGG-16 in the SMOTE run
# (run_vgg_v2/best_hyperparameters.json), so SMOTE is the only thing that changes.
DEFAULT_HP = {
    "lr": 3.8396292998041685e-05, "weight_decay": 2.9204338471814107e-05, "batch_size": 16,
    "dropout": 0.1798695128633439, "cbam_reduction": 16, "optimizer": "adam",
    "label_smoothing": 0.006505159298527952,
}


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# %% Data
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


class RandomRot90:
    """Rotate by 0/90/180/270 degrees. Kolams keep their class under these rotations."""
    def __call__(self, img):
        k = random.randint(0, 3)
        return img.rotate(90 * k, expand=True) if k else img


@lru_cache(maxsize=None)
def get_norm(model_name: str):
    """Mean/std the pretrained backbone expects (ImageNet values for EfficientNet, 0.5 for ViT)."""
    try:
        m = timm.create_model(model_name, pretrained=False, num_classes=0)
        c = timm.data.resolve_model_data_config(m)
        return tuple(c["mean"]), tuple(c["std"])
    except Exception:
        return IMAGENET_MEAN, IMAGENET_STD


def build_transforms(img_size: int, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    train_tf = transforms.Compose([
        RandomRot90(),
        transforms.RandomResizedCrop(img_size, scale=(0.75, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    # Resize the whole image (no centre crop) so the edges of the kolam are kept
    eval_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    return train_tf, eval_tf


def make_splits(cfg: Config):
    base = datasets.ImageFolder(cfg.data_dir)
    targets = np.array(base.targets)
    idx = np.arange(len(targets))
    trainval_idx, test_idx = train_test_split(
        idx, test_size=cfg.test_size, stratify=targets, random_state=cfg.seed)
    train_idx, val_idx = train_test_split(
        trainval_idx, test_size=cfg.val_size, stratify=targets[trainval_idx],
        random_state=cfg.seed)
    return base.classes, base.samples, targets, train_idx, val_idx, test_idx


class RecordsDataset(Dataset):
    """A list of (image_path, label) pairs; used for real and SMOTE images alike."""
    def __init__(self, records, transform):
        self.records, self.transform = records, transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        path, label = self.records[i]
        return self.transform(default_loader(path)), label


def make_loaders(cfg, train_records, val_records, test_records, batch_size):
    train_tf, eval_tf = build_transforms(cfg.img_size, *get_norm(cfg.model_name))
    pin = torch.cuda.is_available()
    common = dict(num_workers=cfg.num_workers, pin_memory=pin)
    train_loader = DataLoader(RecordsDataset(train_records, train_tf), batch_size=batch_size,
                              shuffle=True, drop_last=len(train_records) > batch_size, **common)
    val_loader = DataLoader(RecordsDataset(val_records, eval_tf), batch_size=batch_size * 2,
                            shuffle=False, **common)
    test_loader = DataLoader(RecordsDataset(test_records, eval_tf), batch_size=batch_size * 2,
                             shuffle=False, **common)
    return train_loader, val_loader, test_loader


# %% SMOTE (training split only)
def load_resized(path, size):
    with Image.open(path) as im:
        return np.asarray(im.convert("RGB").resize((size, size), Image.BILINEAR), dtype=np.uint8)


def build_smote_images(cfg, train_records, n_classes, cache_dir):
    """
    Pixel-level SMOTE on the training images.

    For each class smaller than the largest class, repeat until it reaches the
    largest class's count:
      - pick a random training image x_i of that class,
      - pick one of its k nearest neighbours x_j (same class, Euclidean pixel distance),
      - create x_new = x_i + lambda * (x_j - x_i), with lambda ~ Uniform(0, 1).
    Returns a list of (path, label) for the synthetic images.
    """
    labels = np.array([y for _, y in train_records])
    counts = np.bincount(labels, minlength=n_classes)
    target = int(counts.max())
    meta = {"img_size": cfg.img_size, "seed": cfg.seed, "k": cfg.smote_k,
            "train_files": sorted(p for p, _ in train_records)}
    marker = cache_dir / "smote_meta.json"
    if marker.exists():
        saved = json.load(open(marker))
        if all(saved.get(k) == v for k, v in meta.items()) and \
                all(Path(p).exists() for p, _ in saved["synthetic"]):
            print(f"  reusing {len(saved['synthetic'])} cached SMOTE images from {cache_dir}")
            return [tuple(r) for r in saved["synthetic"]]
    shutil.rmtree(cache_dir, ignore_errors=True)
    cache_dir.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(cfg.seed)
    synthetic = []
    for c in range(n_classes):
        need = target - int(counts[c])
        if need <= 0:
            continue
        paths_c = [p for p, y in train_records if y == c]
        if len(paths_c) < 2:
            raise ValueError(f"Class {c} has fewer than 2 training images; SMOTE needs at least 2.")
        X = np.stack([load_resized(p, cfg.img_size) for p in paths_c]).astype(np.float32)
        flat = X.reshape(len(X), -1)
        k = min(cfg.smote_k, len(X) - 1)
        neighbours = NearestNeighbors(n_neighbors=k + 1).fit(flat).kneighbors(
            flat, return_distance=False)[:, 1:]        # drop column 0 (the image itself)
        out_c = cache_dir / f"class_{c:02d}"
        out_c.mkdir(parents=True, exist_ok=True)
        for n in range(need):
            i = rng.integers(len(X))
            j = neighbours[i, rng.integers(k)]
            lam = rng.random()
            new = X[i] + lam * (X[j] - X[i])
            path = out_c / f"smote_{n:05d}.jpg"
            Image.fromarray(np.clip(np.rint(new), 0, 255).astype(np.uint8)).save(path, quality=95)
            synthetic.append((str(path), c))
        print(f"  class {c}: {counts[c]} real + {need} synthetic = {target}")
        del X, flat
    json.dump({**meta, "synthetic": synthetic}, open(marker, "w"))
    return synthetic


def plot_smote_distribution(classes, before, after, path):
    x = np.arange(len(classes))
    w = 0.4
    fig, ax = plt.subplots(figsize=(max(8, len(classes) * 1.2), 4.5))
    ax.bar(x - w / 2, before, w, label="Original training set")
    ax.bar(x + w / 2, after, w, label="After SMOTE")
    ax.set_xticks(x, classes, rotation=35, ha="right")
    ax.set_ylabel("Training images")
    ax.set_title("Training class distribution before and after SMOTE")
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def plot_smote_examples(classes, synthetic, path, per_class=4, seed=0):
    rng = random.Random(seed)
    by_class = {}
    for p, y in synthetic:
        by_class.setdefault(y, []).append(p)
    rows = sorted(by_class)
    if not rows:
        return
    fig, axes = plt.subplots(len(rows), per_class, figsize=(per_class * 2.2, len(rows) * 2.3),
                             squeeze=False)
    for r, c in enumerate(rows):
        picks = rng.sample(by_class[c], min(per_class, len(by_class[c])))
        for k in range(per_class):
            ax = axes[r][k]
            ax.axis("off")
            if k < len(picks):
                ax.imshow(Image.open(picks[k]))
        axes[r][0].set_title(classes[c], fontsize=9, loc="left")
    fig.suptitle("Examples of SMOTE-generated training images")
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)


# %% Model: backbone (EfficientNet / VGG / ViT) + CBAM
class ChannelAttention(nn.Module):
    """'What' to attend to: re-weights feature channels."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, hidden, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1, bias=False),
        )

    def forward(self, x):
        avg = self.mlp(F.adaptive_avg_pool2d(x, 1))
        mx = self.mlp(F.adaptive_max_pool2d(x, 1))
        return x * torch.sigmoid(avg + mx)


class SpatialAttention(nn.Module):
    """'Where' to attend to: re-weights spatial positions."""
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)

    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx, _ = x.max(dim=1, keepdim=True)
        return x * torch.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))


class CBAM(nn.Module):
    """Channel attention followed by spatial attention (Woo et al., 2018)."""
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        # Run attention in float32: under fp16 autocast the channel-attention MLP can
        # overflow (fp16 max is about 65,500) and turn the whole forward pass into NaN
        with torch.autocast(device_type=x.device.type, enabled=False):
            return self.sa(self.ca(x.float()))


class KolamNet(nn.Module):
    """
    Backbone + (optional) CBAM + classifier.

    CNN backbones (e.g. efficientnet_b4, vgg16) output their last convolutional feature
    map (B, C, H, W); CBAM is applied to it, followed by global average pooling. For VGG,
    the original fully connected layers (fc6/fc7) are dropped and replaced by this head.

    ViT backbones (e.g. vit_base_patch16_224) output tokens (B, 1 + H*W, C): one CLS
    token plus one token per image patch. With CBAM, the patch tokens are rearranged
    into their H x W grid (14 x 14 for 224-pixel images with 16-pixel patches), CBAM is
    applied to that grid, and the grid is average-pooled. Without CBAM, the standard
    ViT classifier input (the CLS token) is used.
    """
    def __init__(self, n_classes, model_name="efficientnet_b4", pretrained=True,
                 use_cbam=True, cbam_reduction=16, dropout=0.3, img_size=None):
        super().__init__()
        self.is_vit = model_name.startswith(("vit_", "deit", "eva", "beit"))
        if self.is_vit:
            kwargs = {"img_size": img_size} if img_size else {}
            self.backbone = timm.create_model(model_name, pretrained=pretrained,
                                              num_classes=0, **kwargs)
            self.prefix = self.backbone.num_prefix_tokens          # 1 CLS token
            self.grid = self.backbone.patch_embed.grid_size         # (14, 14)
        else:
            # CNN: use the last convolutional feature map (B, C, H, W) from forward_features.
            self.backbone = timm.create_model(model_name, pretrained=pretrained,
                                              num_classes=0, global_pool="")
            # VGG's fully connected fc6/fc7 layers ("pre_logits") are not used: CBAM needs the
            # convolutional feature map, and the new classifier head replaces them.
            if hasattr(self.backbone, "pre_logits"):
                self.backbone.pre_logits = nn.Identity()
        c = self.backbone.num_features
        self.use_cbam = use_cbam
        self.cbam = CBAM(c, cbam_reduction) if use_cbam else nn.Identity()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(c, n_classes)

    def forward(self, x):
        if self.is_vit:
            tokens = self.backbone.forward_features(x)              # (B, 1 + H*W, C)
            if not self.use_cbam:
                return self.fc(self.dropout(tokens[:, 0]))          # CLS token
            patches = tokens[:, self.prefix:]                       # (B, H*W, C)
            b, n, c = patches.shape
            feats = patches.transpose(1, 2).reshape(b, c, *self.grid)   # (B, C, H, W)
        else:
            feats = self.backbone.forward_features(x)               # (B, C, H, W)
        feats = self.cbam(feats)
        return self.fc(self.dropout(self.pool(feats).flatten(1)))


# %% Training utilities
def make_scaler(device):
    enabled = device.type == "cuda"
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):   # older PyTorch
        return torch.cuda.amp.GradScaler(enabled=enabled)


def run_epoch(model, loader, criterion, device, optimizer=None, scaler=None, max_grad_norm=1.0):
    training = optimizer is not None
    model.train(training)
    use_amp = device.type == "cuda"
    total_loss, n_seen, bad_batches = 0.0, 0, 0
    y_true, y_pred = [], []
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(x)
            loss = criterion(logits.float(), y)          # loss in float32, not fp16

            if training:
                optimizer.zero_grad(set_to_none=True)
                if not torch.isfinite(loss):              # skip a NaN/inf batch instead of
                    bad_batches += 1                      # letting it corrupt the weights
                    continue
                scaler.scale(loss).backward()
                if max_grad_norm and max_grad_norm > 0:
                    scaler.unscale_(optimizer)            # unscale so clipping sees true gradient sizes
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                scaler.step(optimizer)
                scaler.update()

            total_loss += loss.item() * x.size(0)
            n_seen += x.size(0)
            y_true.append(y.cpu())
            y_pred.append(logits.argmax(1).cpu())

    if n_seen == 0:                                       # every batch failed
        return {"loss": float("nan"), "acc": 0.0, "f1": 0.0, "bad_batches": bad_batches}
    y_true, y_pred = torch.cat(y_true).numpy(), torch.cat(y_pred).numpy()
    return {
        "loss": total_loss / n_seen,
        "acc": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "bad_batches": bad_batches,
    }


def fit(model, train_loader, val_loader, hp, device, max_epochs, patience, trial=None,
        max_grad_norm=1.0):
    """Train with early stopping on validation loss; restores the best weights."""
    criterion = nn.CrossEntropyLoss(label_smoothing=hp["label_smoothing"])
    opt_cls = torch.optim.AdamW if hp["optimizer"] == "adamw" else torch.optim.Adam
    optimizer = opt_cls(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min",
                                                           factor=0.5, patience=3)
    scaler = make_scaler(device)

    history = {k: [] for k in ["train_loss", "val_loss", "train_acc", "val_acc",
                               "train_f1", "val_f1", "lr"]}
    best_loss, best_state, best_epoch, wait = float("inf"), None, 0, 0

    for epoch in range(max_epochs):
        tr = run_epoch(model, train_loader, criterion, device, optimizer, scaler, max_grad_norm)
        va = run_epoch(model, val_loader, criterion, device)

        # Divergence check: NaN validation loss, or more than 10% of training batches were NaN
        diverged = (not math.isfinite(va["loss"])) or tr["bad_batches"] > 0.1 * len(train_loader)
        if diverged:
            msg = (f"training diverged at epoch {epoch + 1} "
                   f"({tr['bad_batches']} NaN training batches, val loss {va['loss']})")
            print("  " + msg)
            if trial is not None:
                raise optuna.TrialPruned(msg)         # Optuna skips this trial and moves on
            if best_state is None:
                raise RuntimeError(msg + " before any valid epoch - lower the learning rate")
            break                                     # final model: keep the last good weights

        scheduler.step(va["loss"])

        for split, m in (("train", tr), ("val", va)):
            for k in ("loss", "acc", "f1"):
                history[f"{split}_{k}"].append(m[k])
        history["lr"].append(optimizer.param_groups[0]["lr"])
        skipped = f" | skipped {tr['bad_batches']} NaN batches" if tr["bad_batches"] else ""
        print(f"  epoch {epoch + 1:3d} | train loss {tr['loss']:.4f} acc {tr['acc']:.3f} "
              f"F1 {tr['f1']:.3f} | val loss {va['loss']:.4f} acc {va['acc']:.3f} "
              f"F1 {va['f1']:.3f}{skipped}")

        if va["loss"] < best_loss - 1e-4:
            best_loss, best_epoch, wait = va["loss"], epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1

        if trial is not None:
            trial.report(va["f1"], epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        if wait >= patience:
            print(f"  early stopping at epoch {epoch + 1} (best epoch {best_epoch + 1})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history, best_epoch


@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    y_true, y_pred, probs = [], [], []
    for x, y in loader:
        logits = model(x.to(device))
        p = logits.float().softmax(1).cpu()
        probs.append(p)
        y_pred.append(p.argmax(1))
        y_true.append(y)
    return torch.cat(y_true).numpy(), torch.cat(y_pred).numpy(), torch.cat(probs).numpy()


# %% Optuna
def suggest_hp(trial, cfg):
    return {
        "lr": trial.suggest_float("lr", 1e-5, cfg.lr_max, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "batch_size": trial.suggest_categorical("batch_size", list(cfg.batch_sizes)),
        "dropout": trial.suggest_float("dropout", 0.1, 0.5),
        "cbam_reduction": trial.suggest_categorical("cbam_reduction", [8, 16, 32]),
        "optimizer": trial.suggest_categorical("optimizer", ["adam", "adamw"]),
        "label_smoothing": trial.suggest_float("label_smoothing", 0.0, 0.1),
    }


def build_model(cfg, hp, n_classes, device):
    return KolamNet(n_classes, cfg.model_name, cfg.pretrained, cfg.use_cbam,
                    hp["cbam_reduction"], hp["dropout"], img_size=cfg.img_size).to(device)


def run_study(cfg, n_classes, train_records, val_records, test_records, device, out):
    study = optuna.create_study(
        study_name="kolamnet",
        direction="maximize",
        storage=f"sqlite:///{out / 'optuna.db'}",   # resumable if the session disconnects
        load_if_exists=True,
        # TPE starts guided search after a few random trials (its default of 10 would make
        # short studies pure random search)
        sampler=optuna.samplers.TPESampler(seed=cfg.seed, n_startup_trials=max(3, cfg.n_trials // 3)),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3),
    )

    def objective(trial):
        set_seed(cfg.seed + trial.number)
        hp = suggest_hp(trial, cfg)
        print(f"\nTrial {trial.number}: {hp}")
        train_loader, val_loader, _ = make_loaders(cfg, train_records, val_records, test_records,
                                                   hp["batch_size"])
        model = build_model(cfg, hp, n_classes, device)
        try:
            model, _, _ = fit(model, train_loader, val_loader, hp, device,
                              cfg.tune_epochs, cfg.tune_patience, trial, cfg.max_grad_norm)
        except torch.cuda.OutOfMemoryError:
            del model
            torch.cuda.empty_cache()
            print("  out of GPU memory - trial pruned (reduce --batch_sizes or --img_size)")
            raise optuna.TrialPruned()
        y_true, y_pred, _ = predict(model, val_loader, device)
        return f1_score(y_true, y_pred, average="macro", zero_division=0)

    done = len([t for t in study.trials if t.state.is_finished()])
    remaining = max(cfg.n_trials - done, 0)
    if remaining:
        # catch: one crashed trial is recorded as FAIL and the search continues
        study.optimize(objective, n_trials=remaining, catch=(RuntimeError, ValueError, TypeError))
    return study


# %% Plots
def plot_history(history, best_epoch, path):
    epochs = np.arange(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, key, title in zip(axes, ["loss", "acc", "f1"], ["Loss", "Accuracy", "Macro-F1"]):
        ax.plot(epochs, history[f"train_{key}"], label="train")
        ax.plot(epochs, history[f"val_{key}"], label="validation")
        ax.axvline(best_epoch + 1, color="grey", ls="--", lw=1, label="restored epoch")
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax.grid(alpha=0.3)
    axes[0].legend()
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def plot_confusion(y_true, y_pred, classes, path):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(classes)))
    cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    for ax, mat, title, fmt in ((axes[0], cm, "Counts", "d"),
                                (axes[1], cm_norm, "Row-normalised (recall)", ".2f")):
        im = ax.imshow(mat, cmap="Blues")
        ax.grid(False)
        ax.set_xticks(range(len(classes)), classes, rotation=45, ha="right")
        ax.set_yticks(range(len(classes)), classes)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.set_title(title)
        thresh = mat.max() / 2
        for i in range(len(classes)):
            for j in range(len(classes)):
                ax.text(j, i, format(mat[i, j], fmt), ha="center", va="center",
                        color="white" if mat[i, j] > thresh else "black", fontsize=9)
        fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def plot_per_class(y_true, y_pred, classes, path):
    p, r, f, s = precision_recall_fscore_support(
        y_true, y_pred, labels=range(len(classes)), zero_division=0)
    x = np.arange(len(classes))
    w = 0.27
    fig, ax = plt.subplots(figsize=(max(7, len(classes) * 1.3), 4.5))
    ax.bar(x - w, p, w, label="Precision")
    ax.bar(x, r, w, label="Recall")
    ax.bar(x + w, f, w, label="F1")
    ax.set_xticks(x, [f"{c}\n(n={n})" for c, n in zip(classes, s)], fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.set_title("Per-class test metrics")
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def plot_optuna(study, out):
    from optuna.visualization import matplotlib as ovm
    complete = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    plots = [("optuna_history.png", ovm.plot_optimization_history)]
    if len(complete) >= 2:
        plots.append(("optuna_param_importance.png", ovm.plot_param_importances))
    for name, fn in plots:
        try:
            # Optuna's plots switch matplotlib to the ggplot style; keep that change local
            with plt.style.context("default"):
                ax = fn(study)
            fig = ax.figure if hasattr(ax, "figure") else ax[0].figure
            fig.set_size_inches(8, 5)
            fig.tight_layout()
            fig.savefig(out / name, dpi=150)
            plt.close(fig)
        except Exception as e:   # plotting should never kill a finished run
            print(f"Could not create {name}: {e}")


# %% Main
def main(cfg: Config):
    set_seed(cfg.seed)
    out = Path(cfg.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    mean, std = get_norm(cfg.model_name)
    print(f"Backbone: {cfg.model_name} | image size {cfg.img_size} | CBAM: {cfg.use_cbam} | "
          f"normalisation mean {mean} std {std}")

    classes, samples, targets, train_idx, val_idx, test_idx = make_splits(cfg)
    n_classes = len(classes)
    print("Classes:", classes)
    for name, idx in (("train", train_idx), ("val", val_idx), ("test", test_idx)):
        print(f"  {name:5s} {len(idx):5d} images | per class: "
              f"{np.bincount(targets[idx], minlength=n_classes).tolist()}")
    json.dump({"classes": classes, "train": train_idx.tolist(), "val": val_idx.tolist(),
               "test": test_idx.tolist()}, open(out / "splits.json", "w"))

    train_records = [samples[i] for i in train_idx]
    val_records = [samples[i] for i in val_idx]
    test_records = [samples[i] for i in test_idx]

    # 1. SMOTE on the training split only
    if cfg.use_smote:
        print("\nBalancing the training split with SMOTE")
        cache_dir = Path(cfg.smote_cache) if cfg.smote_cache else \
            Path(tempfile.gettempdir()) / f"kolam_smote_{cfg.img_size}_{cfg.seed}"
        synthetic = build_smote_images(cfg, train_records, n_classes, cache_dir)
        before = np.bincount(targets[train_idx], minlength=n_classes)
        train_records = train_records + synthetic
        after = np.bincount([y for _, y in train_records], minlength=n_classes)
        print(f"  training set: {int(before.sum())} real + {len(synthetic)} synthetic = "
              f"{int(after.sum())} | per class: {after.tolist()}")
        plot_smote_distribution(classes, before, after, out / "smote_class_distribution.png")
        plot_smote_examples(classes, synthetic, out / "smote_examples.png", seed=cfg.seed)
    else:
        print("\nSMOTE disabled: training on the original (imbalanced) split")

    # 2. Hyperparameter search
    if cfg.n_trials > 0:
        study = run_study(cfg, n_classes, train_records, val_records, test_records, device, out)
        study.trials_dataframe().to_csv(out / "optuna_trials.csv", index=False)
        states = [t.state.name for t in study.trials]
        print(f"\nTrials: {states.count('COMPLETE')} complete, {states.count('PRUNED')} pruned, "
              f"{states.count('FAIL')} failed")
        if states.count("COMPLETE") == 0:
            best_hp = dict(DEFAULT_HP)
            print("No trial completed (all pruned or diverged); using DEFAULT_HP:", best_hp)
        else:
            best_hp = dict(study.best_params)
            print(f"Best trial {study.best_trial.number}: val macro-F1 = {study.best_value:.4f}")
            print("Best hyperparameters:", best_hp)
            plot_optuna(study, out)
    else:
        best_hp = dict(DEFAULT_HP)
        print("Skipping tuning; using DEFAULT_HP:", best_hp)
    json.dump(best_hp, open(out / "best_hyperparameters.json", "w"), indent=2)

    # 3. Final training with the best hyperparameters
    print("\nTraining final model")
    set_seed(cfg.seed)
    train_loader, val_loader, test_loader = make_loaders(
        cfg, train_records, val_records, test_records, best_hp["batch_size"])
    model = build_model(cfg, best_hp, n_classes, device)
    t0 = time.time()
    model, history, best_epoch = fit(model, train_loader, val_loader, best_hp, device,
                                     cfg.final_epochs, cfg.patience,
                                     max_grad_norm=cfg.max_grad_norm)
    train_minutes = (time.time() - t0) / 60
    torch.save({"state_dict": model.state_dict(), "classes": classes,
                "hyperparameters": best_hp, "config": asdict(cfg)}, out / "kolamnet_best.pt")
    pd.DataFrame(history).rename_axis("epoch").to_csv(out / "history.csv")

    # 4. Test evaluation (the only time the test split is used; real images only)
    y_true, y_pred, _ = predict(model, test_loader, device)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    pw, rw, fw, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted",
                                                    zero_division=0)
    metrics = {
        "test_accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": p, "macro_recall": r, "macro_f1": f,
        "weighted_precision": pw, "weighted_recall": rw, "weighted_f1": fw,
        "n_test_images": int(len(y_true)),
        "n_train_images": len(train_records),
        "restored_epoch": best_epoch + 1,
        "epochs_run": len(history["train_loss"]),
        "train_minutes": round(train_minutes, 1),
        "use_cbam": cfg.use_cbam,
        "use_smote": cfg.use_smote,
    }
    report = classification_report(y_true, y_pred, labels=range(n_classes),
                                   target_names=classes, digits=3, zero_division=0)
    json.dump(metrics, open(out / "test_metrics.json", "w"), indent=2)
    (out / "classification_report.txt").write_text(report)
    print("\nTest metrics:", json.dumps(metrics, indent=2))
    print(report)

    # 5. Plots
    plot_history(history, best_epoch, out / "training_curves.png")
    plot_confusion(y_true, y_pred, classes, out / "confusion_matrix.png")
    plot_per_class(y_true, y_pred, classes, out / "per_class_metrics.png")
    print(f"All outputs saved to {out.resolve()}")
    return model, metrics


def parse_args() -> Config:
    parser = argparse.ArgumentParser(description="Kolam classifier (EfficientNet / VGG / ViT + CBAM) with SMOTE and Optuna")
    for f in fields(Config):
        default = f.default
        if isinstance(default, bool):
            parser.add_argument(f"--{f.name}", default=default,
                                type=lambda s: s.lower() in ("1", "true", "yes", "y"))
        elif isinstance(default, tuple):
            parser.add_argument(f"--{f.name}", default=default, type=int, nargs="+")
        else:
            parser.add_argument(f"--{f.name}", default=default, type=type(default))
    args, _ = parser.parse_known_args()   # parse_known_args also works inside Jupyter
    values = vars(args)
    values["batch_sizes"] = tuple(values["batch_sizes"])
    return Config(**values)


if __name__ == "__main__":
    main(parse_args())

Writing kolamnet_optuna.py


In [3]:
!python -u kolamnet_optuna.py --data_dir "/kaggle/input/datasets/skuvilan/kolam-dataset/Kolam_Project" --out_dir run_vgg_nosmote --model_name vgg16 --img_size 224 --use_smote false --n_trials 0 --num_workers 4

Device: cuda
Backbone: vgg16 | image size 224 | CBAM: True | normalisation mean (0.485, 0.456, 0.406) std (0.229, 0.224, 0.225)
Classes: ['Artifical Kolam', 'Digital Art Kolam', 'Mandala Kolam', 'Peakcock Kolam', 'Pushpa Kolam_Pookalam', 'Rangoli Kolam', 'Sikku_Pulli Kolam_Geometric Patterns', 'Traditional Kolam_Colourful', 'Traditional Kolam_Dominant White Colour']
  train  4040 images | per class: [136, 373, 1364, 328, 230, 286, 395, 398, 530]
  val    1011 images | per class: [34, 93, 341, 82, 58, 72, 99, 100, 132]
  test   1263 images | per class: [42, 117, 427, 103, 72, 89, 123, 124, 166]

SMOTE disabled: training on the original (imbalanced) split
Skipping tuning; using DEFAULT_HP: {'lr': 3.8396292998041685e-05, 'weight_decay': 2.9204338471814107e-05, 'batch_size': 16, 'dropout': 0.1798695128633439, 'cbam_reduction': 16, 'optimizer': 'adam', 'label_smoothing': 0.006505159298527952}

Training final model
model.safetensors: 100%|██████████████████████| 553M/553M [00:04<00:00, 126MB

In [4]:
!zip -qr run_vgg_nosmote.zip run_vgg_nosmote